# 02 — Feature Engineering & Cross-Sectional Target

**Part A** computes standard technical indicators per stock.

**Part B** builds the target the "right" way for this problem: instead of predicting
whether a stock's price goes up or down in absolute terms (close to a random walk, and
shown not to work in earlier iterations of this project), we rank each stock's 5-day
forward return against the other 19 stocks *on the same date* and label the top half as
outperformers. This cancels out market-wide moves (index rallies/selloffs, macro news,
RBI announcements) that hit every stock at once, and isolates stock-specific signal —
which is what technical indicators can plausibly capture.

## Part A — Feature engineering

In [1]:
import pandas as pd
import numpy as np
from ta import add_all_ta_features
import warnings
warnings.filterwarnings('ignore')

In [2]:
TICKERS = [
    "RELIANCE.NS", "TCS.NS", "INFY.NS", "HDFCBANK.NS", "ICICIBANK.NS",
    "HINDUNILVR.NS", "ITC.NS", "SBIN.NS", "BHARTIARTL.NS", "KOTAKBANK.NS",
    "LT.NS", "AXISBANK.NS", "ASIANPAINT.NS", "MARUTI.NS", "SUNPHARMA.NS",
    "TITAN.NS", "ULTRACEMCO.NS", "WIPRO.NS", "NESTLEIND.NS", "TATAMOTORS.NS",
]

raw = {}
for ticker in TICKERS:
    fname = ticker.replace(".", "_")
    df = pd.read_csv(f"../data/{fname}_raw.csv", index_col="Date", parse_dates=True)
    raw[ticker] = df
    print(f"{ticker}: {df.shape}")

RELIANCE.NS: (1853, 5)
TCS.NS: (1853, 5)
INFY.NS: (1853, 5)
HDFCBANK.NS: (1853, 5)
ICICIBANK.NS: (1853, 5)
HINDUNILVR.NS: (1853, 5)
ITC.NS: (1853, 5)
SBIN.NS: (1853, 5)
BHARTIARTL.NS: (1853, 5)
KOTAKBANK.NS: (1853, 5)
LT.NS: (1853, 5)
AXISBANK.NS: (1853, 5)
ASIANPAINT.NS: (1853, 5)
MARUTI.NS: (1853, 5)
SUNPHARMA.NS: (1853, 5)
TITAN.NS: (1853, 5)
ULTRACEMCO.NS: (1853, 5)
WIPRO.NS: (1853, 5)
NESTLEIND.NS: (1853, 5)
TATAMOTORS.NS: (1853, 5)


In [3]:
def build_features(df):
    """Compute technical indicators and keep a fixed subset of columns.
    No Target here — target is cross-sectional and built in Part B below."""
    data = df.copy()
    data = add_all_ta_features(data, open="Open", high="High", low="Low", close="Close",
                                volume="Volume", fillna=True)

    cols_to_keep = [
        'Close',
        'trend_sma_fast', 'trend_sma_slow', 'trend_ema_fast',       # Trend
        'trend_macd', 'trend_macd_signal', 'trend_macd_diff',       # MACD
        'momentum_rsi',                                              # RSI
        'volatility_bbh', 'volatility_bbl', 'volatility_bbw',       # Bollinger Bands
        'volatility_atr',                                            # ATR
        'volume_obv',                                                # Volume
    ]
    data = data[cols_to_keep]

    data['return_1d']  = data['Close'].pct_change(1)
    data['return_5d']  = data['Close'].pct_change(5)
    data['return_10d'] = data['Close'].pct_change(10)

    return data

In [4]:
featured = {}
for ticker, df in raw.items():
    featured[ticker] = build_features(df)
    print(f"{ticker}: {featured[ticker].shape}")

RELIANCE.NS: (1853, 16)


TCS.NS: (1853, 16)


INFY.NS: (1853, 16)


HDFCBANK.NS: (1853, 16)


ICICIBANK.NS: (1853, 16)


...

TITAN.NS: (1853, 16)


ULTRACEMCO.NS: (1853, 16)


WIPRO.NS: (1853, 16)


NESTLEIND.NS: (1853, 16)


TATAMOTORS.NS: (1853, 16)


## Part B — Cross-sectional target construction

For every date, rank all 20 stocks by their 5-day forward return. `Target = 1` if a
stock's rank is in the top half that day, else `0`. Because this is a rank (not an
absolute threshold), the label is balanced by construction — roughly 50/50 for every
stock, regardless of whether the overall market went up or down that period.

In [5]:
fwd_returns = pd.DataFrame({
    ticker: featured[ticker]['Close'].pct_change(5).shift(-5)
    for ticker in TICKERS
})

rank_pct = fwd_returns.rank(axis=1, pct=True)

# NOTE: (rank_pct > 0.5).astype(int) alone would silently turn NaN ranks (the trailing
# ~5 trading days per stock, where there isn't yet a future close to compute a forward
# return against) into Target=0 instead of NaN -- pandas evaluates `NaN > 0.5` as False,
# and casting False to int gives 0. That would corrupt real trailing rows with a fake
# label instead of letting the dropna() below remove them. Explicitly preserve NaN here
# so those rows are actually dropped, not mislabeled.
target_all = (rank_pct > 0.5).astype(float)
target_all[rank_pct.isna()] = np.nan

In [6]:
for ticker in TICKERS:
    featured[ticker]['Target'] = target_all[ticker]
    featured[ticker].dropna(inplace=True)
    featured[ticker]['Target'] = featured[ticker]['Target'].astype(int)

### Target balance sanity check — should be close to ~50/50 per stock

In [7]:
for ticker in TICKERS:
    balance = featured[ticker]['Target'].value_counts(normalize=True).sort_index()
    first_date = featured[ticker].index.min().date()
    last_date = featured[ticker].index.max().date()
    print(f"{ticker:15s} rows={featured[ticker].shape[0]:4d}  "
          f"Target=0: {balance.get(0, 0):.2%}  Target=1: {balance.get(1, 0):.2%}  "
          f"valid-Target range: {first_date} -> {last_date}")

RELIANCE.NS     rows=1838  Target=0: 51.31%  Target=1: 48.69%  valid-Target range: 2019-01-15 -> 2026-06-23
TCS.NS          rows=1838  Target=0: 52.50%  Target=1: 47.50%  valid-Target range: 2019-01-15 -> 2026-06-23
INFY.NS         rows=1838  Target=0: 48.26%  Target=1: 51.74%  valid-Target range: 2019-01-15 -> 2026-06-23
HDFCBANK.NS     rows=1838  Target=0: 48.59%  Target=1: 51.41%  valid-Target range: 2019-01-15 -> 2026-06-23
ICICIBANK.NS    rows=1838  Target=0: 47.82%  Target=1: 52.18%  valid-Target range: 2019-01-15 -> 2026-06-23
HINDUNILVR.NS   rows=1838  Target=0: 54.46%  Target=1: 45.54%  valid-Target range: 2019-01-15 -> 2026-06-23
ITC.NS          rows=1838  Target=0: 55.17%  Target=1: 44.83%  valid-Target range: 2019-01-15 -> 2026-06-23
SBIN.NS         rows=1838  Target=0: 46.90%  Target=1: 53.10%  valid-Target range: 2019-01-15 -> 2026-06-23
BHARTIARTL.NS   rows=1838  Target=0: 46.35%  Target=1: 53.65%  valid-Target range: 2019-01-15 -> 2026-06-23
KOTAKBANK.NS    rows=1838  T

## Save featured data

In [8]:
for ticker in TICKERS:
    fname = ticker.replace(".", "_")
    featured[ticker].to_csv(f"../data/{fname}_features.csv")
    print(f"Saved ../data/{fname}_features.csv")

Saved ../data/RELIANCE_NS_features.csv
Saved ../data/TCS_NS_features.csv


Saved ../data/INFY_NS_features.csv
Saved ../data/HDFCBANK_NS_features.csv
Saved ../data/ICICIBANK_NS_features.csv
Saved ../data/HINDUNILVR_NS_features.csv
Saved ../data/ITC_NS_features.csv
Saved ../data/SBIN_NS_features.csv
Saved ../data/BHARTIARTL_NS_features.csv
Saved ../data/KOTAKBANK_NS_features.csv
Saved ../data/LT_NS_features.csv


Saved ../data/AXISBANK_NS_features.csv


Saved ../data/ASIANPAINT_NS_features.csv


Saved ../data/MARUTI_NS_features.csv
Saved ../data/SUNPHARMA_NS_features.csv


Saved ../data/TITAN_NS_features.csv
Saved ../data/ULTRACEMCO_NS_features.csv
Saved ../data/WIPRO_NS_features.csv
Saved ../data/NESTLEIND_NS_features.csv
Saved ../data/TATAMOTORS_NS_features.csv
